### 1. Connect Google Drive and Configure Paths
We mount Google Drive to access the large dataset files and define the paths to the compressed JSONL files.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd

# Mount Google Drive
drive.mount("/content/drive")

# Define data directory
DATA_DIR = Path("/content/drive/MyDrive/amazon-reviews-2023-analysis/data")

ARCHIVO_REVIEWS = DATA_DIR / "Beauty_and_Personal_Care.jsonl.gz"
ARCHIVO_METADATA = DATA_DIR / "meta_Beauty_and_Personal_Care.jsonl.gz"

# Verify files
print("Reviews file exists?:", ARCHIVO_REVIEWS.exists())
print("Metadata file exists?:", ARCHIVO_METADATA.exists())

### 2. Loading Data Samples
Because these files are several gigabytes in size, we use the `chunksize` parameter to load only the first 5,000 records for initial analysis.

In [ ]:
# Load 5,000 reviews
reviews_df = next(
    pd.read_json(ARCHIVO_REVIEWS, lines=True, compression="gzip", chunksize=5000)
)

# Load 5,000 metadata records
metadata_df = next(
    pd.read_json(ARCHIVO_METADATA, lines=True, compression="gzip", chunksize=5000)
)

print(f"Reviews loaded: {reviews_df.shape}")
print(f"Metadata loaded: {metadata_df.shape}")

### 3. Data Exploration and Column Summary
We define a helper function to inspect the data types, missing values, and overall structure of the loaded samples.

In [ ]:
def resumen_columnas(df):
    return pd.DataFrame({
        "columna": df.columns,
        "tipo": df.dtypes.astype(str).values,
        "cantidad_nulos": df.isna().sum().values,
        "porcentaje_nulos": (df.isna().mean().values * 100).round(2)
    })

print("Resumen de Reviews:")
display(resumen_columnas(reviews_df))

print("\nResumen de Metadata:")
display(resumen_columnas(metadata_df))

### 4. Merging Reviews and Metadata
We link the user reviews with the product information using the `parent_asin` identifier to create a comprehensive summary of top products.

In [ ]:
# 1. Aggregate reviews by product
reviews_summary = (
    reviews_df
    .groupby("parent_asin")
    .agg(
        review_count=("rating", "count"),
        average_rating=("rating", "mean"),
        helpful_votes=("helpful_vote", "sum")
    )
    .reset_index()
)

# 2. Select key metadata columns and drop duplicates
products_info = metadata_df[
    ["parent_asin", "title", "store", "average_rating", "rating_number", "price"]
].drop_duplicates("parent_asin")

# 3. Merge dataframes
final_analysis = reviews_summary.merge(products_info, on="parent_asin", how="left")

print("Top products by review count in the sample:")
display(final_analysis.sort_values("review_count", ascending=False).head(10))

# Sección nueva

# AMAZON REVIEWS
